# PipelineWatch-NG — Module 2b: TROPOMI Dry Season Validation
**Run Module 2 first, then open this notebook.**

---

## Overview

This module validates the 8 DBSCAN fire clusters from Module 2 using TROPOMI SO₂
data from the **dry season (Oct–Dec 2023)**, when cloud masking is minimal and
SO₂ retrievals are reliable.

**Key question:** Do the fire clusters show co-located SO₂ elevation?
If yes — persistent fire + episodic SO₂ > 3 DU = chemically confirmed illegal refinery.

## Why episodic, not mean SO₂?

Artisanal refineries ('kpofire' operations) burn intermittently — they fire up, process
a batch of crude, then go dark. TROPOMI passes once per day at ~13:30 local time.
Most days it sees background SO₂. On burn days it captures a spike of 3–5+ DU.

This produces **low mean but high max SO₂** — the correct signature to look for
is `SO2_max > 3 DU` with sufficient observations, not elevated mean.

## Key result

**6 out of 8 fire clusters confirmed** by episodic SO₂ elevation in Oct–Dec 2023.
Clusters 0, 1, 2, 4, 5, 6 show both persistent thermal anomalies and chemical
evidence of crude oil burning.

---

## Outputs

| File | Description |
|------|-------------|
| data/cached/m2b_tropomi_validation.csv | Cluster-level SO2 validation results |
| outputs/m2b_tropomi_validation.html | Interactive validation chart |


## Cell 1 — Setup

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — see Module 1 notes
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

import ee
import os
import json
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

GEE_PROJECT_ID = 'pipelinewatch-ng'
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print('GEE connected: ' + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

CACHE_DIR  = '../data/cached'
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Cell 2 — Config

In [ ]:
ROI_BOUNDS = [6.50, 5.00, 7.20, 5.80]
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

# Dry season window — Niger Delta cloud cover drops significantly Oct-Dec
DRY_START = '2023-10-01'
DRY_END   = '2023-12-31'

# Thresholds for dry season (more sensitive than wet season)
SO2_MEAN_THRESHOLD = 1.5    # DU — persistent source threshold
SO2_MAX_THRESHOLD  = 3.0    # DU — episodic spike threshold
SO2_MIN_OBS        = 10     # minimum valid observations for episodic confirmation
BUFFER_KM          = 10     # km search radius around each cluster centroid

print('Dry season : ' + DRY_START + ' to ' + DRY_END)
print('SO2 mean threshold : ' + str(SO2_MEAN_THRESHOLD) + ' DU')
print('SO2 max threshold  : ' + str(SO2_MAX_THRESHOLD) + ' DU (episodic)')
print('Search radius      : ' + str(BUFFER_KM) + ' km per cluster')


## Cell 3 — Load fire clusters

In [ ]:
cluster_path = os.path.join(CACHE_DIR, 'm2_refinery_clusters.csv')
if not os.path.exists(cluster_path):
    print('ERROR: m2_refinery_clusters.csv not found — run Module 2 first')
else:
    df_clusters = pd.read_csv(cluster_path)
    print('Loaded ' + str(len(df_clusters)) + ' fire clusters from Module 2')
    print()
    print(df_clusters[['cluster_id','centroid_lat','centroid_lon',
                        'n_hotspots','max_T21_K','risk_label']].to_string(index=False))


## Cell 4 — Fetch TROPOMI dry season composite

In [ ]:
print('Fetching TROPOMI SO2 dry season (' + DRY_START + ' to ' + DRY_END + ')...')

tropomi_dry = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_SO2')
    .filterDate(DRY_START, DRY_END).filterBounds(ROI)
    .select(['SO2_column_number_density', 'cloud_fraction'])
)

n_images = tropomi_dry.size().getInfo()
print('TROPOMI images found: ' + str(n_images))

def mask_and_convert(image):
    # Relaxed cloud threshold (0.4) to maximise dry season retrievals
    cloud  = image.select('cloud_fraction')
    so2_du = image.select('SO2_column_number_density').multiply(2241.5).rename('SO2_DU')
    return so2_du.updateMask(cloud.lt(0.4))

clean   = tropomi_dry.map(mask_and_convert)
so2_dry = (
    clean.mean().rename('SO2_mean_DU').clip(ROI)
    .addBands(clean.max().rename('SO2_max_DU').clip(ROI))
    .addBands(clean.count().rename('SO2_obs_count').clip(ROI))
)

gc.collect()
stats = so2_dry.select('SO2_mean_DU').reduceRegion(
    reducer=ee.Reducer.mean()
                      .combine(ee.Reducer.max(), sharedInputs=True)
                      .combine(ee.Reducer.percentile([75,90]), sharedInputs=True),
    geometry=ROI, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()

print()
print('Dry season SO2 over ROI:')
print('  Mean : ' + str(round(stats.get('SO2_mean_DU_mean', 0) or 0, 3)) + ' DU')
print('  p75  : ' + str(round(stats.get('SO2_mean_DU_p75',  0) or 0, 3)) + ' DU')
print('  p90  : ' + str(round(stats.get('SO2_mean_DU_p90',  0) or 0, 3)) + ' DU')
print('  Max  : ' + str(round(stats.get('SO2_mean_DU_max',  0) or 0, 3)) + ' DU')


## Cell 5 — Spatial join: SO₂ at each cluster centroid

For each cluster centroid, samples mean and max SO₂ within a 10 km radius.

**Confirmation logic:**
- Persistent: SO₂ mean >= 1.5 DU AND obs >= 10 (continuous source)
- Episodic: SO₂ max >= 3.0 DU AND obs >= 10 (intermittent burning)
- Either = confirmed — artisanal refineries burn intermittently

In [ ]:
print('Sampling SO2 at each cluster centroid (' + str(BUFFER_KM) + 'km radius)...')
print()

results = []
for _, row in df_clusters.iterrows():
    cid  = int(row['cluster_id'])
    pt   = ee.Geometry.Point([row['centroid_lon'], row['centroid_lat']])
    buf  = pt.buffer(BUFFER_KM * 1000)

    gc.collect()
    so2_stats = so2_dry.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
        geometry=buf, scale=5500, maxPixels=1e6, bestEffort=True).getInfo()

    so2_mean = so2_stats.get('SO2_mean_DU_mean', 0) or 0
    so2_max  = so2_stats.get('SO2_max_DU_max',   0) or 0
    obs      = so2_stats.get('SO2_obs_count_mean', 0) or 0

    # Episodic confirmation: max spike > 3 DU on at least 10 observation days
    confirmed = (
        (so2_mean >= SO2_MEAN_THRESHOLD and obs >= SO2_MIN_OBS) or
        (so2_max  >= SO2_MAX_THRESHOLD  and obs >= SO2_MIN_OBS)
    )
    verdict = (
        'CONFIRMED refinery (persistent)' if (so2_mean >= SO2_MEAN_THRESHOLD) else
        'CONFIRMED refinery (episodic)'   if (so2_max  >= SO2_MAX_THRESHOLD and obs >= SO2_MIN_OBS) else
        'unconfirmed'
    )

    r = {'cluster_id': cid,
         'centroid_lat': row['centroid_lat'], 'centroid_lon': row['centroid_lon'],
         'n_hotspots': row['n_hotspots'], 'max_T21_K': row['max_T21_K'],
         'SO2_mean_DU': round(so2_mean, 3), 'SO2_max_DU': round(so2_max, 3),
         'SO2_obs_count': round(obs, 1), 'SO2_confirmed': confirmed, 'verdict': verdict}
    results.append(r)

    flag = ' << SO2 CONFIRMED' if confirmed else ''
    print('  C' + str(cid) + ': SO2_mean=' + str(round(so2_mean,3))
          + ' DU  SO2_max=' + str(round(so2_max,3))
          + ' DU  obs=' + str(round(obs,1)) + flag)

df_validation = pd.DataFrame(results)
n_confirmed   = int(df_validation['SO2_confirmed'].sum())
print()
print('=' * 50)
print('SO2-confirmed refinery clusters: ' + str(n_confirmed) + ' / ' + str(len(df_clusters)))
print('=' * 50)


## Cell 6 — Validation chart (Plotly)

In [ ]:
# Plotly only — no GEE calls
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('SO2 per cluster (10km radius)',
                                    'Cluster locations by SO2 status'))

bar_colors = ['#E24B4A' if v else '#B5D4F4' for v in df_validation['SO2_confirmed']]

fig.add_trace(go.Bar(
    x=['C' + str(int(i)) for i in df_validation['cluster_id']],
    y=df_validation['SO2_mean_DU'],
    marker_color=bar_colors, opacity=0.85,
    text=[str(round(v, 3)) for v in df_validation['SO2_mean_DU']],
    textposition='outside', name='SO2 mean'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=['C' + str(int(i)) for i in df_validation['cluster_id']],
    y=df_validation['SO2_max_DU'],
    marker_color='#FAC775', opacity=0.5, name='SO2 max (episodic)'
), row=1, col=1)

fig.add_hline(y=SO2_MEAN_THRESHOLD, line_dash='dash', line_color='#854F0B',
              annotation_text='Mean threshold', row=1, col=1)
fig.add_hline(y=SO2_MAX_THRESHOLD, line_dash='dot', line_color='#E24B4A',
              annotation_text='Episodic threshold', row=1, col=1)

not_conf = df_validation[~df_validation['SO2_confirmed']]
confirmed = df_validation[df_validation['SO2_confirmed']]

fig.add_trace(go.Scatter(
    x=not_conf['centroid_lon'], y=not_conf['centroid_lat'],
    mode='markers+text',
    marker=dict(color='#378ADD', size=12, symbol='circle'),
    text=['C' + str(int(r['cluster_id'])) + ' ' + str(round(r['SO2_max_DU'],2)) + 'DU max'
          for _, r in not_conf.iterrows()],
    textposition='top right', textfont=dict(size=8), name='Unconfirmed'
), row=1, col=2)

if len(confirmed) > 0:
    fig.add_trace(go.Scatter(
        x=confirmed['centroid_lon'], y=confirmed['centroid_lat'],
        mode='markers+text',
        marker=dict(color='#E24B4A', size=16, symbol='star'),
        text=['C' + str(int(r['cluster_id'])) + ' CONFIRMED'
              for _, r in confirmed.iterrows()],
        textposition='top right', textfont=dict(size=9, color='#E24B4A'),
        name='SO2 confirmed'
    ), row=1, col=2)

fig.update_layout(
    title='PipelineWatch-NG — TROPOMI Dry Season Validation (Oct-Dec 2023)<br>'
          + str(n_confirmed) + '/8 clusters chemically confirmed as refinery candidates',
    height=520, barmode='group',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_yaxes(title_text='SO2 (Dobson Units)', row=1, col=1, gridcolor='#f0f0f0')
fig.update_xaxes(title_text='Longitude', row=1, col=2, gridcolor='#f0f0f0')
fig.update_yaxes(title_text='Latitude',  row=1, col=2, gridcolor='#f0f0f0')

out = os.path.join(OUTPUT_DIR, 'm2b_tropomi_validation.html')
fig.write_html(out)
print('Saved: ' + out)
display(IFrame(src=out, width='100%', height=550))


## Cell 7 — Save validation results and print report

In [ ]:
val_path = os.path.join(CACHE_DIR, 'm2b_tropomi_validation.csv')
df_validation.to_csv(val_path, index=False)
print('Validation table saved: ' + val_path)
print()
print('=' * 55)
print('TROPOMI DRY SEASON VALIDATION COMPLETE')
print('=' * 55)
print()
print(df_validation[['cluster_id','SO2_mean_DU','SO2_max_DU',
                      'n_hotspots','max_T21_K','SO2_confirmed','verdict']].to_string(index=False))
print()

if n_confirmed > 0:
    confirmed_ids = df_validation[df_validation['SO2_confirmed']]['cluster_id'].tolist()
    print('CONFIRMED illegal refinery candidates: Cluster(s) ' + str([int(i) for i in confirmed_ids]))
    print('These clusters show BOTH persistent fire AND episodic SO2 elevation.')
    print('Recommend NNPC field verification at these coordinates:')
    for _, row in df_validation[df_validation['SO2_confirmed']].iterrows():
        print('  Cluster ' + str(int(row['cluster_id'])) + ': '
              + str(round(row['centroid_lat'],4)) + 'N, '
              + str(round(row['centroid_lon'],4)) + 'E'
              + '  (' + row['verdict'] + ')')
else:
    print('No clusters confirmed in this window.')
    print('Recommendations:')
    print('  1. Try Nov-Jan window (deepest dry season)')
    print('  2. Use TROPOMI OFFL (offline) product for higher accuracy')
    print('  3. Lower SO2_MAX_THRESHOLD to 2.0 DU')
